## Tarefa Prática para os Métodos Iterativos de Solução de Sistemas de Equações Lineares

In [87]:
import numpy as np

#### Método de Jacobi-Richardson

In [88]:
def jacobi_richardson(A, b, itMax, epslon):
    n = A.shape[0]

    L = np.zeros((n, n))
    D = np.zeros((n, n))
    U = np.zeros((n, n))

    #definindo LDU
    for i in range(n):
        for j in range(n):
            # Matriz triangular inferior
            if i > j:
                L[i, j] = A[i, j]
            # Matriz diagonal
            elif i == j:
                D[i, j] = A[i, j]
            # Matriz triangular superior
            else:
                U[i, j] = A[i, j]

    #chute inicial
    x = np.zeros([n])
    erro = 0
    i = 0
    for i in range(itMax):
        xNovo = np.linalg.inv(D) @ (b - (L + U) @ x)
        erro = np.linalg.norm(xNovo - x)
        x = xNovo
        
        if erro < epslon:
            break
    
    return x, i

#### Método de Gauss-Seidel

In [89]:
def gauss_seidel(A, b, itMax, epslon):
    n = A.shape[0]

    L = np.zeros((n, n))
    U = np.zeros((n, n))
    I = np.eye((n))

    #definindo L* e U*
    for i in range(n):
        for j in range(n):
            # Matriz triangular inferior
            if i > j:
                L[i, j] = A[i, j] / A[i, i]
            # Matriz triangular infoerior
            elif i < j:
                U[i, j] = A[i, j] / A[i, i]

    for i in range(n):
        b[i] = b[i] / A[i, i]

    #chute inicial
    x = np.zeros([n])
    erro = 0
    i = 0
    for i in range(itMax):
        xNovo = - np.linalg.inv((L + I)) @ U @ x + np.linalg.inv((L + I)) @ b
        erro = np.linalg.norm(xNovo - x)
        x = xNovo

        if erro < epslon:
            break
    
    return x, i


#### Método de SOR

In [90]:
def SOR(A, b, itMax, epslon, omega):

    n = A.shape[0]

    L = np.zeros((n, n))
    U = np.zeros((n, n))
    I = np.eye((n))

    #definindo L* e U*
    for i in range(n):
        for j in range(n):
            # Matriz triangular inferior
            if i > j:
                L[i, j] = A[i, j] / A[i, i]
            # Matriz triangular infoerior
            elif i < j:
                U[i, j] = A[i, j] / A[i, i]

    for i in range(n):
        b[i] = b[i] / A[i, i]

    #construindo A*
    A = L + U + I

    #chute inicial
    xSOR = np.zeros(n)
    xNovoSOR = np.zeros(n)
    xGS = np.zeros(n)
    erro = 0

    i = 0
    for i in range(itMax):
        xNovoGS = - np.linalg.inv((L + I)) @ U @ x + np.linalg.inv((L + I)) @ b
        xGS = xNovoGS.copy()
        
        for j in range(n):
            soma1 = 0
            for k in range(j):
                soma1 += A[j, k] * xGS[k]
            soma2 = 0
            for k in range(j + 1, n):
                soma2 += A[j, k] * xSOR[k]
            xNovoSOR[j] = (1 - omega) * xSOR[j] + omega * (b[j] - soma1 - soma2)
        
        erro = np.linalg.norm(xNovoSOR - xSOR)
        xSOR = xNovoSOR.copy()

        if erro < epslon:
            break
    
    return xSOR, i


1. Considere o seguinte sistema de equações lineares:

$$
\begin{cases}
15x_1 + 5x_2 - 5x_3 = 30 \\
4x_1 + 10x_2 + x_3 = 23 \\
2x_1 - 2x_2 + 8x_3 = -10
\end{cases}
$$

Utilizando os scripts desenvolvidos, obtenha a solução numérica para esse sistema de equações lineares pelos métodos de Jacobi-Richardson, Gauss-Seidel e SOR, considerando os critérios de parada $\varepsilon = 10^{-5}$ e $k_{\text{max}} = 30$, e utilizando, para o método SOR, os valores de $\omega = 1$ e $\omega = 1.5$.

Após obter os resultados, apresente-os em uma tabela que compare as soluções obtidas e o número de iterações de cada método.

In [91]:
A = np.array([
    [15.0, 5.0, -5.0],
    [4.0,10.0, 1.0],
    [2.0, 2.0, 8.0]
])
b = np.array([30.0, 23.0, -10.0])

itMax = 30
epslon = 1e-5
omega = [1, 1.5]

x_result = []
it_result = []

x, i = jacobi_richardson(A.copy(), b.copy(), itMax, epslon)
x_result.append(x)
it_result.append(i)

x, i = gauss_seidel(A.copy(), b.copy(), itMax, epslon)
x_result.append(x)
it_result.append(i)

x, i = SOR(A.copy(), b.copy(), itMax, epslon, omega[0])
x_result.append(x)
it_result.append(i)

x, i = SOR(A.copy(), b.copy(), itMax, epslon, omega[1])
x_result.append(x)
it_result.append(i)

metodos = ["jacobi", "gauss", "SOR w=1", "SOR w=1.5"]
print(f"{'Método':<15} | {'Solução':<37} | {'Iter.':<20}")
print("-" * 64)

for i in range(len(metodos)):
    nome = metodos[i]
    vetor = x_result[i]
    it = it_result[i]

    print(f"{nome:<15} | {vetor} | {it:<4} |")


Método          | Solução                               | Iter.               
----------------------------------------------------------------
jacobi          | [ 0.59259492  2.25925679 -1.96296282] | 13   |
gauss           | [ 0.59259269  2.25925922 -1.96296298] | 7    |
SOR w=1         | [ 0.59259259  2.25925926 -1.96296296] | 3    |
SOR w=1.5       | [ 0.59259621  2.25925997 -1.96296285] | 23   |
